# MARS-SOHO Phase 1 — train-only replay reconstruction gate

Run **one dataset per session**. This notebook never extracts or opens test features. It compares a matched exact-replay oracle with shared-Gaussian, heterogeneous spherical, support-aware and shuffled-support reconstruction. SRQ is not enabled in Phase 1.

In [ ]:
# === Edit DATASET_KEY only when starting a new independent session. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
DATASET_KEY = 'cifar100'  # cifar100, cub200, or imagenetr
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_ROOT = '/content/mars_soho_phase1_features'
OUTPUT_ROOT = '/content/mars_soho_phase1_outputs'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_CONFIG_SHA256 = 'cac28a17c5ddaac4a6a42af60d4501d794a7f52d09af82f81a44b6c428067f3a'
EXPECTED_RUNNER_SHA256 = '578cae2c620e9d3f0499abe55535508d2904edcc3c3c8f9f01d5552deb79847e'

In [ ]:
# Fresh immutable checkout and source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG = 'configs/mars_soho_phase1_train_only.json'
RUNNER = 'tools/mars_soho_phase1.py'
assert sha(CONFIG) == EXPECTED_CONFIG_SHA256
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip()
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('MARS-SOHO PHASE-1 SOURCE CHECK: PASS')

In [ ]:
# Download the verified frozen ViT checkpoint and the selected dataset only.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
HANDLES = {'cifar100':'zaphat206/cifar-100','cub200':'zaphat206/cub-200-2011','imagenetr':'zaphat206/imagenet-r'}
assert DATASET_KEY in HANDLES
DATASET_ROOT = kagglehub.dataset_download(HANDLES[DATASET_KEY])
print('dataset:', DATASET_KEY, DATASET_ROOT)

In [ ]:
# Extract frozen TRAIN features only. Progress is printed by task; test.pt must stay absent.
protocol = json.loads(Path(CONFIG).read_text())
dataset = protocol['datasets'][DATASET_KEY]
cache = Path(FEATURE_CACHE_ROOT) / DATASET_KEY
if not (cache/'train.pt').is_file():
    command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATASET_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],'--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_mars_{DATASET_KEY}','--dataset',dataset['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit','--seed','2025','--num-classes',str(dataset['num_classes']),'--num-tasks',str(dataset['num_tasks']),'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START. Wait for one progress line per task.', flush=True)
    subprocess.run(command, check=True)
assert (cache/'train.pt').is_file() and (cache/'metadata.json').is_file()
assert not (cache/'test.pt').exists(), 'FAIL: test.pt became visible'
print('TRAIN CACHE READY:', cache, '| test.pt absent')

In [ ]:
# Mathematical, state, checkpoint and tiny train-only runner gate.
command = [sys.executable,'-B','-m','pytest','-q','tests/test_mars_soho_math.py','tests/test_mars_soho_learner.py','tests/test_mars_soho_phase1.py']
completed = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(completed.stdout, flush=True)
assert completed.returncode == 0, 'Correctness gate failed.'
print('MARS-SOHO PHASE-1 CORRECTNESS GATE: PASS')

## Locked train-only run
The runner prints `START/RESTORED`, one `TASK` line per continual stage, and `DONE`. It first reuses one exact-replay execution to select ridge lambda, then searches four shared/heterogeneous rank-shrinkage configurations. Support-aware and shuffled controls inherit the chosen heterogeneous configuration so only allocation changes. Completed units resume while this runtime disk remains available.

In [ ]:
# Start/resume Phase 1. No held-out feature or label is opened.
command = [sys.executable,'-u',RUNNER,'--config',CONFIG,'--dataset-key',DATASET_KEY,'--feature-cache-dir',str(cache),'--output-root',OUTPUT_ROOT,'--device','cuda']
print(f'STARTING MARS-SOHO PHASE 1: {DATASET_KEY}', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'elapsed={(time.time()-started)/60:.1f} minutes | return_code={completed.returncode}', flush=True)
assert completed.returncode == 0, 'Runner failed; return the full traceback without editing the grid.'
RESULT_PATH = Path(OUTPUT_ROOT)/DATASET_KEY/'phase1_results.json'
assert RESULT_PATH.is_file()
print('PHASE-1 PROCESS COMPLETE')

In [ ]:
# Inspect outer-validation evidence and diagnostics. This is not test accuracy.
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
payload = json.loads(RESULT_PATH.read_text())
rows = [{'method':method,'outer_validation_AIA':score} for method,score in payload['outer_mean_aia'].items()]
table = pd.DataFrame(rows).sort_values('outer_validation_AIA', ascending=False)
display(table)
print('selected lambda:', payload['selected_ridge_lambda'])
print('selected reconstruction:', json.dumps(payload['selected_reconstruction'], indent=2))
print('gates:', json.dumps(payload['gates'], indent=2))
ax = sns.barplot(data=table, x='method', y='outer_validation_AIA')
ax.tick_params(axis='x', rotation=30); ax.set_title(f'{DATASET_KEY}: train-only outer-validation AIA')
plt.tight_layout(); plt.show()
risk_rows=[]
for ridx,result in enumerate(payload['outer_validation']['support_aware']):
    for task,diag in enumerate(result['task_diagnostics'],1):
        for class_id,risk in diag.get('boundary_risk',{}).items(): risk_rows.append({'replicate':ridx,'task':task,'class_id':int(class_id),'risk':risk,'allocation':diag.get('pseudo_allocation',{}).get(class_id,diag.get('pseudo_allocation',{}).get(str(class_id)))})
if risk_rows:
    risk_df=pd.DataFrame(risk_rows); display(risk_df.describe())
    sns.scatterplot(data=risk_df,x='risk',y='allocation',hue='replicate',alpha=.6); plt.title('Support risk vs fixed-budget allocation'); plt.show()

In [ ]:
# Export evidence only. Frozen per-sample feature cache is deliberately excluded.
from google.colab import files
evidence = Path(OUTPUT_ROOT)/DATASET_KEY
shutil.copy2(CONFIG, evidence/'locked_config.json')
shutil.copy2(RUNNER, evidence/'locked_runner.py')
archive = shutil.make_archive(f'/content/mars_soho_phase1_{DATASET_KEY}_train_only','zip',root_dir=evidence)
print('artifact:', archive, 'bytes=',Path(archive).stat().st_size, 'sha256=',sha(archive))
files.download(archive)